# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

# RAG From Scratch: Advanced Indexing

Explores advanced indexing strategies that go beyond simple chunk-and-embed. This notebook covers **Multi-Representation Indexing** — storing summaries in the vector store for retrieval while keeping raw documents in a separate docstore for generation.

**Prerequisites**: Run the environment setup and indexing cells from previous notebooks first. Chunking (the pre-phase of indexing) is covered in the [Part 1-4 notebook](../part_1_4/part_1_4.ipynb).

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# Suppress all warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Load environment variables from the .env file into the process
load_dotenv()

try: 
    # Map .env variables to the keys LangChain/LangSmith expects at runtime
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")   # Enable LangSmith tracing
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")         # LangSmith authentication key
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")         # LangSmith project name for grouping traces
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")       # LangSmith API endpoint URL
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")             # Mistral AI LLM/embedding API key
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")                           # HuggingFace access token
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0"                           # Required User-Agent header for WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Environment Initialization

Loads `.env` variables for LangSmith tracing, Mistral API, HuggingFace token, and User-Agent. Suppresses warnings.

### Part 12: Multi-Representation Indexing

Stores **summaries** in the vector store and **raw documents** in a docstore, linked by `doc_id`. During retrieval, summaries are matched via similarity search, then the corresponding full documents are fetched from the docstore for generation.

This approach gives the best of both worlds: concise summaries improve retrieval precision, while full documents provide complete context for the LLM.

The process is to take a document, create a summary of it, and then use that summary to answer questions about the document.
Whereas the raw the document is stored in the docstore, the summary is stored in the vector store to be used for similarity search with the user 
query, then retrieve the relevant documents and use them to answer the user query.

and you can see it more detailed in the following diagram: 

```text
┌─────────────────────────────────────────────────────────┐
│                    INDEXING PHASE                       │
│                                                         │
│  Raw Doc ──► Summarize ──► Store Summary in Vectorstore │
│     │                                                   │
│     └──────────────────► Store Raw Doc in Docstore      │
│                          (linked by same doc_id)        │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│                    RETRIEVAL PHASE                      │
│                                                         │
│  User Query ──► Similarity Search on Summaries          │
│                        │                                │
│                        ▼                                │
│               Find matching doc_id                      │
│                        │                                │
│                        ▼                                │
│               Fetch Raw Doc from Docstore               │
│                        │                                │
│                        ▼                                │
│               Return Raw Doc to LLM                     │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the first blog post (LLM-powered agents)
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

# Load the second blog post (human data quality) and append to the same list
loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())  # Now docs contains 2 full Document objects

#### Load Source Documents

Fetches two blog posts using `WebBaseLoader`:
1. Lilian Weng's post on LLM-powered agents.
2. Lilian Weng's post on human data quality.

Both are loaded as full `Document` objects and combined into a single list.

In [ ]:
# Inspect metadata of the first document (source URL, title, etc.)
# Uncomment the line below to see the full page content:
# docs[0].page_content
docs[0].metadata

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'title': "LLM Powered Autonomous Agents | Lil'Log",
 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\

#### Generate Summaries

Uses a summarization chain (`ChatMistralAI` with `mistral-medium-latest`) to produce concise summaries of each document. `chain.batch()` processes both documents in parallel with `max_concurrency=5`.

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI

# Summarization chain: extract page_content → summarize with LLM → parse to string
chain = (
    {"doc": lambda x: x.page_content}                                            # Extract the document text
    | ChatPromptTemplate.from_template("Summarize the following document: \n\n{doc}")  # Summarization prompt
    | ChatMistralAI(model="mistral-medium-latest", max_retries=0)                 # Generate summary
    | StrOutputParser()                                                            # Parse LLM output to string
)

# Process both documents in parallel (max_concurrency=5)
summaries = chain.batch(docs, {"max_concurrency": 5})

#### Set Up MultiVectorRetriever

Creates the dual-store architecture:
- **Vector store** (`Chroma`, `"summaries"` collection) — Stores summary embeddings for similarity search.
- **Byte store** (`InMemoryByteStore`) — Stores raw documents keyed by `doc_id`.
- **`MultiVectorRetriever`** — Links both stores; searches summaries, returns raw docs.

Each summary `Document` is tagged with a `doc_id` that maps to the corresponding raw document in the byte store. Both stores are populated via `add_documents()` and `mset()`.

In [ ]:
from langchain_core.stores import InMemoryByteStore
from langchain_mistralai import MistralAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

# ==========================
# Vector Store: stores summary embeddings for similarity search
# ==========================
vectorstore = Chroma(
    collection_name = "summaries",                  # Collection name for summary embeddings
    embedding_function = MistralAIEmbeddings(),     # Mistral embedding model
    persist_directory = "./chroma_db",              # Persist in the current directory
)

# ==========================
# Byte Store: stores raw (full) documents keyed by doc_id
# ==========================
store = InMemoryByteStore()     # In-memory key-value store for raw documents
id_key = "doc_id"               # Metadata key that links summaries to raw docs

# ==========================
# MultiVectorRetriever: searches summaries, returns raw documents
# ==========================
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,    # Searches against summary embeddings
    byte_store=store,           # Fetches raw documents by doc_id
    id_key=id_key,              # The metadata key used to link both stores
)

# Generate unique IDs for each raw document
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Create summary Documents tagged with their corresponding doc_id
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# Populate the vector store with summary embeddings
retriever.vectorstore.add_documents(summary_docs)

# Populate the byte store with raw documents (linked by doc_id)
retriever.docstore.mset(list(zip(doc_ids, docs)))

#### Test: Similarity Search on Summaries

Runs a direct similarity search on the vector store with the query *"Memory in Agents"*. Returns the matched **summary** document (not the raw document). This confirms the summaries are properly indexed.

In [ ]:
# Test: direct similarity search on the vector store (returns SUMMARY documents)
query = "Memory in Agents"
sub_docs = vectorstore.similarity_search(query, k=1)  # Top-1 most similar summary
sub_docs  # This returns the summary, NOT the raw document

[Document(id='c00308d4-7106-46e5-9419-8de2dcbacc65', metadata={'doc_id': '3a451ed8-fd05-4113-801f-07d3baa0071e'}, page_content='### **Summary of *LLM Powered Autonomous Agents* by Lilian Weng (June 2023)**\n\nThis article explores the architecture, capabilities, and challenges of **LLM-powered autonomous agents**, where large language models (LLMs) act as the "brain" of agents capable of planning, memory management, and tool use. Below is a structured summary:\n\n---\n\n### **1. Agent System Overview**\nAn LLM-powered autonomous agent consists of three core components:\n- **Planning**: Breaking tasks into subgoals and refining actions via self-reflection.\n- **Memory**: Short-term (in-context learning) and long-term (external vector stores).\n- **Tool Use**: Calling external APIs (e.g., search engines, code execution) to access real-time or proprietary data.\n\n---\n\n### **2. Component Breakdown**\n\n#### **A. Planning**\n- **Task Decomposition**:\n  - **Chain of Thought (CoT)**: Step

#### Test: Full Retrieval via MultiVectorRetriever

Invokes the `MultiVectorRetriever` with the same query. Unlike direct similarity search, this returns the **full raw document** from the byte store (linked by `doc_id`). Prints the first 500 characters to verify the raw content.

In [ ]:
# Test: full retrieval via MultiVectorRetriever (returns RAW documents from byte store)
# The retriever searches summaries, finds the matching doc_id, then fetches the full raw doc
retrieved_docs = retriever.invoke(query, n_results = 1)

# Preview the first 500 characters of the raw document to verify it's the full content
retrieved_docs[0].page_content[0:500]

"\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three:"

### Part 13: RAPTOR

**RAPTOR** (Recursive Abstractive Processing for Tree-Organized Retrieval) builds a hierarchical tree of document summaries at multiple levels of abstraction. This enables retrieval at different granularities — from fine-grained chunks to high-level topic summaries.

> Implementation to be added in a future update.